In [1]:
import sqlite3

In [2]:
class Block:
    def __init__(self, hexString, id, view, desc, img):
        self.hexString = hexString
        self.id = id
        self.view = view
        self.desc = desc
        self.img = img

    @classmethod
    def get_by_hex(cls, hexString, conn):
        cur = conn.cursor()
        cur.execute(
            "SELECT hexString, id, view, desc, img FROM BLOCKS WHERE hexString = ?",
            (hexString,)
        )
        row = cur.fetchone()
        if row:
            return cls(*row)
        return None

    @classmethod
    def get_all(cls, conn):
        cur = conn.cursor()
        cur.execute("SELECT hexString, id, view, desc, img FROM BLOCKS")
        rows = cur.fetchall()
        return [cls(*row) for row in rows]

    def __str__(self):
        return f"Block({self.hexString}, {self.id}, {self.desc})"

In [3]:
class Source:
    def __init__(self, id, ip_addr, country_code):
        self.id = id
        self.ip_addr = ip_addr
        self.country_code = country_code

    @classmethod
    def get_by_id(cls, id, conn):
        cur = conn.cursor()
        cur.execute("SELECT id, ip_addr, country_code FROM SOURCES WHERE id = ?", (id,))
        row = cur.fetchone()
        if row:
            return cls(*row)
        return None

    @classmethod
    def get_all(cls, conn):
        cur = conn.cursor()
        cur.execute("SELECT id, ip_addr, country_code FROM SOURCES")
        rows = cur.fetchall()
        return [cls(*row) for row in rows]

    def __str__(self):
        return f"Source({self.id}, {self.ip_addr}, {self.country_code})"

In [4]:
class Person:
    def __init__(self, id, name, addr):
        self.id = id
        self.name = name
        self.addr = addr

    @classmethod
    def get_by_id(cls, id, conn):
        cur = conn.cursor()
        cur.execute("SELECT id, name, addr FROM PERSONS WHERE id = ?", (id,))
        row = cur.fetchone()
        if row:
            return cls(*row)
        return None

    @classmethod
    def get_all(cls, conn):
        cur = conn.cursor()
        cur.execute("SELECT id, name, addr FROM PERSONS")
        rows = cur.fetchall()
        return [cls(*row) for row in rows]

    def __str__(self):
        return f"Person({self.id}, {self.name})"

In [5]:
class Vote:
    def __init__(self, block_id, voter_id, timestamp, source_id):
        self.block_id = block_id
        self.voter_id = voter_id
        self.timestamp = timestamp
        self.source_id = source_id

    @classmethod
    def get_by_block_and_voter(cls, block_id, voter_id, conn):
        cur = conn.cursor()
        cur.execute(
            "SELECT block_id, voter_id, timestamp, source_id FROM VOTES WHERE block_id = ? AND voter_id = ?",
            (block_id, voter_id)
        )
        row = cur.fetchone()
        if row:
            return cls(*row)
        return None

    @classmethod
    def get_all(cls, conn):
        cur = conn.cursor()
        cur.execute("SELECT block_id, voter_id, timestamp, source_id FROM VOTES")
        rows = cur.fetchall()
        return [cls(*row) for row in rows]

    @classmethod
    def get_full_details(cls, block_id, voter_id, conn):
        cur = conn.cursor()
        cur.execute("""
            SELECT v.block_id, v.voter_id, v.timestamp, v.source_id,
                   b.view, b.desc,
                   p.name, p.addr,
                   s.ip_addr, s.country_code
            FROM VOTES v
            LEFT JOIN BLOCKS b ON v.block_id = b.hexString
            LEFT JOIN PERSONS p ON v.voter_id = p.id
            LEFT JOIN SOURCES s ON v.source_id = s.id
            WHERE v.block_id = ? AND v.voter_id = ?
        """, (block_id, voter_id))
        row = cur.fetchone()
        if row:
            return {
                'block_id': row[0],
                'voter_id': row[1],
                'timestamp': row[2],
                'source_id': row[3],
                'block_view': row[4],
                'block_desc': row[5],
                'voter_name': row[6],
                'voter_addr': row[7],
                'source_ip': row[8],
                'source_country': row[9]
            }
        return None

    def __str__(self):
        return f"Vote({self.block_id}, {self.voter_id}, {self.timestamp})"

In [6]:
if __name__ == "__main__":
    conn = sqlite3.connect('lab3_blockchain.db')

    print("Усі блоки:")
    for b in Block.get_all(conn):
        print(f"    {b}")

    print("\nУсі особи:")
    for p in Person.get_all(conn):
        print(f"    {p}")

    print("\nУсі джерела:")
    for s in Source.get_all(conn):
        print(f"    {s}")

    print("\nУсі голоси:")
    votes = Vote.get_all(conn)
    for v in votes:
        print(f"    {v}")

    if votes:
        first = votes[0]
        details = Vote.get_full_details(first.block_id, first.voter_id, conn)
        if details:
            print("\nДеталі першого голосу:")
            for key, value in details.items():
                print(f"    {key}: {value}")

Усі блоки:
    Block(0x1a2b3c4d, 1001, First Block)
    Block(0x5e6f7a8b, 1002, Second Block)
    Block(0x9c0d1e2f, 1003, Third Block)

Усі особи:
    Person(1, Michael Roberts)
    Person(2, Emily Thompson)
    Person(3, James Anderson)

Усі джерела:
    Source(1, 192.168.1.100, US)
    Source(2, 10.0.0.50, UK)
    Source(3, 172.16.0.25, DE)
    Source(4, 192.168.1.200, FR)

Усі голоси:
    Vote(0x1a2b3c4d, 1, 2024-01-15 10:30:00)
    Vote(0x1a2b3c4d, 2, 2024-01-15 10:31:00)
    Vote(0x5e6f7a8b, 1, 2024-01-15 10:32:00)
    Vote(0x5e6f7a8b, 3, 2024-01-15 10:33:00)
    Vote(0x9c0d1e2f, 2, 2024-01-15 10:34:00)
    Vote(0x9c0d1e2f, 3, 2024-01-15 10:35:00)
    Vote(0x1a2b3c4d, 1, 2024-01-15 10:30:00)
    Vote(0x1a2b3c4d, 2, 2024-01-15 10:31:00)
    Vote(0x5e6f7a8b, 1, 2024-01-15 10:32:00)
    Vote(0x5e6f7a8b, 3, 2024-01-15 10:33:00)
    Vote(0x9c0d1e2f, 2, 2024-01-15 10:34:00)
    Vote(0x9c0d1e2f, 3, 2024-01-15 10:35:00)

Деталі першого голосу:
    block_id: 0x1a2b3c4d
    voter_id: 1
    

In [7]:
conn.close()